# Construcao do Dataset — QSO

Pipeline completo para construir o HDF5 com espectros de **Quasars (QSOs)** do
eBOSS/SDSS DR17.

**Etapas:**
1. Combinar catalogos NGC + SGC (clustering)
2. Cross-match com o catalogo full
3. Ler espectros do SDSS (com cache local) e calcular S/N
4. Salvar HDF5 estruturado

**Estrutura do HDF5 gerado (`data/processed/QSO/QSOspectra_raw.h5`):**

```
catalog/                          <- arrays flat (N,)
  ├── spec_id           (str)     "QSO_000042"
  │
  ├── plate, mjd, fiberid (int)
  ├── ra, dec           (float)
  ├── is_qso_final      (int)     0/1 — confirma classificacao
  │
  ├── redshift          (float)   Z (target)
  ├── zwarning          (int)
  │
  ├── sn_median         (float)   computado por nos
  ├── sn_median_all     (float)   do pipeline
  │
  # Fotometria optica (MODELMAG vetor [u,g,r,i,z])
  ├── g_mag, r_mag, i_mag, z_mag (float)
  ├── gr_color, ri_color (float)
  │
  # Fotometria infravermelha (WISE)
  ├── w1_mag, w1_w2_color (float)
  │
  # === BONUS: estimativas alternativas de redshift ===
  ├── z_pca, deltachi2_pca, zwarn_pca         <- PCA
  ├── z_vi                                     <- visual (gold standard)
  ├── z_dr12q                                  <- catalogo anterior
  │
  # === BONUS: redshift por linha de emissao ===
  ├── z_lya,    deltachi2_lya,    zwarn_lya    <- Lyman-alpha 1216
  ├── z_civ,    deltachi2_civ,    zwarn_civ    <- C IV 1549
  ├── z_ciii,   deltachi2_ciii,   zwarn_ciii   <- C III] 1909
  ├── z_mgii,   deltachi2_mgii,   zwarn_mgii   <- Mg II 2798
  ├── z_hbeta,  deltachi2_hbeta,  zwarn_hbeta  <- H-beta 4861
  └── z_halpha, deltachi2_halpha, zwarn_halpha <- H-alpha 6563
```

> **QSO e diferente:** o catalogo NAO tem `CHI2`, `ZERR`, `NPIXELS`, `SPECTYPE`,
> `SUBTYPE`. Em compensacao, tem multiplas estimativas de redshift (PCA + por
> linha de emissao). Salvamos todas para comparacao no paper.

> **Argumento pro paper:** QSOs sao notoriamente dificeis pelo fato das linhas
> serem largas. O pipeline tradicional mede z por linha (cada uma com vies);
> nossa CNN aprende a usar todas simultaneamente, superando qualquer
> single-line estimator.


## 1. Imports e Configuracao

In [14]:
import os
import sys
import time
import hashlib
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np
import h5py
from tqdm import tqdm
from astropy.io import fits
from astropy.table import Table
from astropy import units as u
from astropy.coordinates import SkyCoord

# Adiciona raiz do projeto ao sys.path para importar config
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR
while not (PROJECT_ROOT / "config.py").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from config import paths_for, print_info

OBJECT_TYPE = "QSO"
print_info()

paths = paths_for(OBJECT_TYPE)
print(f"\nObjeto: {OBJECT_TYPE}")
print(f"NGC FITS:      {paths['ngc_fits']}")
print(f"SGC FITS:      {paths['sgc_fits']}")
print(f"FULL FITS:     {paths['full_fits']}")
print(f"Combined out:  {paths['combined_fits']}")
print(f"Filtered out:  {paths['filtered_fits']}")
print(f"HDF5 out:      {paths['spectra_h5']}")
print(f"Cache:         {paths['cache_dir']}")

# Cria diretorios necessarios
paths["cache_dir"].mkdir(parents=True, exist_ok=True)
paths["spectra_h5"].parent.mkdir(parents=True, exist_ok=True)
paths["filtered_fits"].parent.mkdir(parents=True, exist_ok=True)


PROJECT_ROOT: /home/thalita/spec_z_ml
DATA_BASE:    /home/thalita/spec_z_ml/data
  → raw:       /home/thalita/spec_z_ml/data/raw
  → processed: /home/thalita/spec_z_ml/data/processed
  → filtered:  /home/thalita/spec_z_ml/data/filtered
LOGS_DIR:     /home/thalita/spec_z_ml/logs
MODELS_DIR:   /home/thalita/spec_z_ml/models
RESULTS_DIR:  /home/thalita/spec_z_ml/results
Ambiente:     LOCAL

Objeto: QSO
NGC FITS:      /home/thalita/spec_z_ml/data/raw/eBOSS_QSO_clustering_data-NGC-vDR16.fits
SGC FITS:      /home/thalita/spec_z_ml/data/raw/eBOSS_QSO_clustering_data-SGC-vDR16.fits
FULL FITS:     /home/thalita/spec_z_ml/data/raw/eBOSS_QSO_full_ALLdata-vDR16.fits
Combined out:  /home/thalita/spec_z_ml/data/raw/eBOSS_QSO_clustering_combined-vDR16.fits
Filtered out:  /home/thalita/spec_z_ml/data/filtered/filtered_eBOSS_QSO.fits
HDF5 out:      /home/thalita/spec_z_ml/data/processed/QSO/QSOspectra_all.h5
Cache:         /home/thalita/spec_z_ml/data/processed/QSO/cache


## 2. Inspecionar colunas dos FITS

Antes de processar, dar uma olhada nas colunas disponiveis em cada FITS.
Util pra saber o que vai pra `catalog/` no HDF5.


In [15]:
def inspect_fits(path, max_cols=20):
    print(f"\n=== {path.name} ===")
    with fits.open(path) as h:
        d = h[1].data
        print(f"  N rows: {len(d):,}")
        names = d.dtype.names
        print(f"  N cols: {len(names)}")
        print(f"  Cols (primeiras {max_cols}): {names[:max_cols]}")

inspect_fits(paths['ngc_fits'])
inspect_fits(paths['sgc_fits'])
inspect_fits(paths['full_fits'], max_cols=30)



=== eBOSS_QSO_clustering_data-NGC-vDR16.fits ===
  N rows: 218,209
  N cols: 9
  Cols (primeiras 20): ('RA', 'DEC', 'Z', 'WEIGHT_FKP', 'WEIGHT_SYSTOT', 'WEIGHT_CP', 'WEIGHT_NOZ', 'NZ', 'QSO_ID')

=== eBOSS_QSO_clustering_data-SGC-vDR16.fits ===
  N rows: 125,499
  N cols: 9
  Cols (primeiras 20): ('RA', 'DEC', 'Z', 'WEIGHT_FKP', 'WEIGHT_SYSTOT', 'WEIGHT_CP', 'WEIGHT_NOZ', 'NZ', 'QSO_ID')

=== eBOSS_QSO_full_ALLdata-vDR16.fits ===
  N rows: 655,521
  N cols: 89
  Cols (primeiras 30): ('RUN', 'CAMCOL', 'FIELD', 'ID', 'RERUN', 'FIBER2MAG', 'RA', 'DEC', 'PSFFLUX', 'PSFFLUX_IVAR', 'EXTINCTION', 'FIBER2FLUX', 'FIBER2FLUX_IVAR', 'MODELFLUX', 'MODELFLUX_IVAR', 'MODELMAG', 'W1_MAG', 'W1_MAG_ERR', 'W1_NANOMAGGIES', 'W1_NANOMAGGIES_IVAR', 'W2_NANOMAGGIES', 'W2_NANOMAGGIES_IVAR', 'OBJID_TARGETING', 'CHUNK', 'TILE', 'LOCATIONID', 'NTILES', 'BOSSTILE_STATUS', 'TRIMMED', 'IDUPLICATE')


## 3. Combinar NGC + SGC

Os catalogos clustering vem separados por hemisferio (NGC/SGC). Concatena
ambos em um unico FITS combinado.


In [16]:
def combine_ngc_sgc(ngc_path, sgc_path, out_path):
    if out_path.exists():
        print(f"Combinado ja existe: {out_path}")
        with fits.open(out_path) as h:
            return Table(h[1].data)

    with fits.open(ngc_path) as h:
        ngc = Table(h[1].data)
    with fits.open(sgc_path) as h:
        sgc = Table(h[1].data)

    print(f"NGC: {len(ngc):,} | SGC: {len(sgc):,}")
    combined = Table(np.hstack([ngc, sgc]))
    combined.write(out_path, format="fits", overwrite=True)
    print(f"Combinado: {len(combined):,} -> {out_path}")
    return combined


clustering = combine_ngc_sgc(paths['ngc_fits'], paths['sgc_fits'], paths['combined_fits'])
print(f"\nColunas: {clustering.colnames[:15]}...")


Combinado ja existe: /home/thalita/spec_z_ml/data/raw/eBOSS_QSO_clustering_combined-vDR16.fits

Colunas: ['RA', 'DEC', 'Z', 'WEIGHT_FKP', 'WEIGHT_SYSTOT', 'WEIGHT_CP', 'WEIGHT_NOZ', 'NZ', 'QSO_ID']...


## 4. Cross-match com catalogo full

O catalogo `clustering` tem RA/DEC mas faltam alguns metadados (magnitudes,
etc.) que estao no catalogo `full`. Fazemos um match angular de 1 arcsec
para anexar essas colunas.


In [17]:
def cross_match(clustering, full_path, out_path, max_sep_arcsec=1.0):
    if out_path.exists():
        print(f"Filtrado ja existe: {out_path}")
        with fits.open(out_path) as h:
            return Table(h[1].data)

    with fits.open(full_path) as h:
        full = Table(h[1].data)
    print(f"Full catalog: {len(full):,} objetos")

    cl_coords   = SkyCoord(ra=clustering['RA'] * u.deg, dec=clustering['DEC'] * u.deg)
    full_coords = SkyCoord(ra=full['RA'] * u.deg,        dec=full['DEC'] * u.deg)

    idx, d2d, _ = cl_coords.match_to_catalog_sky(full_coords)
    mask = d2d < max_sep_arcsec * u.arcsec
    filtered = full[idx[mask]]

    filtered.write(out_path, format="fits", overwrite=True)
    print(f"Match (<{max_sep_arcsec}\"): {len(filtered):,}/{len(clustering):,} -> {out_path}")
    return filtered


filtered = cross_match(clustering, paths['full_fits'], paths['filtered_fits'])

import pandas as pd
pd.set_option("display.max_columns", None)
scalar_cols = [c for c in filtered.colnames if len(filtered[c].shape) <= 1]
df_filtered = filtered[scalar_cols].to_pandas()
df_filtered.head()


Filtrado ja existe: /home/thalita/spec_z_ml/data/filtered/filtered_eBOSS_QSO.fits


,RUN,CAMCOL,FIELD,ID,RERUN,RA,DEC,W1_MAG,W1_MAG_ERR,W1_NANOMAGGIES,W1_NANOMAGGIES_IVAR,W2_NANOMAGGIES,W2_NANOMAGGIES_IVAR,OBJID_TARGETING,CHUNK,NTILES,BOSSTILE_STATUS,TRIMMED,IDUPLICATE,IGEOMETRY,INGROUP,FIRSTGROUP,MULTGROUP,NEXTGROUP,SECTOR,QSO_ID,IMATCH,Z,PLATE,FIBERID,MJD,Z_DR12Q,IS_QSO_DR12Q,Z_DR7Q_SCH,IS_QSO_DR7Q,IS_QSO_FINAL,ZWARN_PCA,DELTACHI2_PCA,Z_HALPHA,ZWARN_HALPHA,DELTACHI2_HALPHA,Z_HBETA,ZWARN_HBETA,DELTACHI2_HBETA,Z_MGII,ZWARN_MGII,Z_CIII,ZWARN_CIII,DELTACHI2_CIII,Z_CIV,ZWARN_CIV,DELTACHI2_CIV,Z_LYA,ZWARN_LYA,DELTACHI2_LYA,DELTACHI2_MGII,MY_CLASS_PQN,RANDOM_SELECT,Z_10K,IS_QSO_10K,PRIM_REC,Z_VI,ZWARNING,Z_PCA,XFOCAL,YFOCAL,PLATESN2,SN_MEDIAN_ALL,SPEC1_G,SPEC1_R,SPEC1_I,SPEC2_G,SPEC2_R,SPEC2_I,sector_SSR,WEIGHT_CP,sector_TSR,COMP_BOSS
0,1889,6,159,159,301,126.252116,46.486392,15.852880,0.033243,455.877533,0.005133,1578.993530,0.000375,1237653589021622431,214,0,16,1,-1,575,-1,-1,-1,-1,523,583496,2,1.646228,441,1,51868,-1.0,-1,1.6407,1,1,0,2539.858909,-1.0,7682,0.0,-1.0,7682,0.0,1.648395,0,1.637178,0,179.517198,1.632813,0,331.234408,-1.000000,7682,0.000000,186.818500,N/A,16959,NaN,16959,16959,NaN,999999,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.954286,1.0,0.930851,0.984043
1,6727,1,33,722,301,114.685532,31.657900,14.919592,0.016766,1076.869629,0.003616,4224.827148,0.000320,1237674365380788946,24,0,0,1,-1,12234,-1,-1,-1,-1,5536,367863,2,1.876441,541,1,51959,-1.0,-1,1.8738,1,1,0,5052.864014,-1.0,7682,0.0,-1.0,7682,0.0,1.875714,0,1.872989,0,337.843924,1.875665,0,258.236355,-1.000000,7682,0.000000,1046.220564,N/A,16959,NaN,16959,16959,NaN,999999,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.900000,1.0,0.948509,0.986450
2,6709,5,21,360,301,116.471522,33.738967,16.192551,0.054679,333.411133,0.003547,1106.376465,0.000332,1237674290218074472,16,0,0,1,-1,11697,-1,-1,-1,-1,4987,406415,2,2.021702,542,1,51993,-1.0,-1,2.0255,1,1,0,7261.225820,-1.0,7682,0.0,-1.0,7682,0.0,2.025166,0,2.014082,0,282.845163,2.016173,0,1995.189712,2.144605,2082,21.985234,210.421124,N/A,16959,NaN,16959,16959,NaN,999999,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.957265,1.0,0.948649,0.991892
3,2074,2,185,207,301,150.527270,56.893562,16.330244,0.046728,293.699066,0.006259,894.565186,0.000487,1237654381444792527,4,0,0,1,-1,6666,-1,-1,-1,-1,4154,681677,2,1.273896,558,1,52317,-1.0,-1,1.2722,1,1,0,2427.871984,-1.0,7682,0.0,-1.0,7682,0.0,1.271858,0,1.273878,0,265.562072,-1.000000,7682,0.000000,-1.000000,7682,0.000000,825.924479,N/A,16959,NaN,16959,16959,NaN,999999,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.940741,1.0,0.924658,0.986301
4,1453,3,45,65,301,247.810965,47.769813,14.666909,0.006710,1359.052612,0.014176,5244.062988,0.001162,1237651714797797441,3,0,0,1,-1,4621,-1,-1,-1,-1,3635,597639,2,1.436251,625,1,52145,-1.0,-1,1.4385,1,1,0,3904.654895,-1.0,7682,0.0,-1.0,7682,0.0,1.432329,0,1.437345,0,307.002552,1.553926,2050,17.259411,-1.000000,7682,0.000000,427.835517,N/A,16959,NaN,16959,16959,NaN,999999,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.944444,1.0,1.000000,1.000000


## 5. Funcao para ler espectros (com cache)

Le um espectro do disco se ja foi baixado, senao baixa do SDSS DR17.
Cache local evita re-download em runs futuros.


In [18]:
def get_cache_path(cache_dir, plate, mjd, fiber):
    """Caminho deterministico do cache (hash MD5)."""
    key = f"{plate}-{mjd}-{fiber}"
    h = hashlib.md5(key.encode()).hexdigest()
    return cache_dir / f"{h}.fits"


def candidate_urls(plate, mjd, fiber):
    """URLs do SDSS DR17 a tentar em ordem.

    QSOs do DR16Q incluem espectros legados (SDSS-I/II, plate < 3500),
    cuja estrutura no SAS e diferente do eBOSS.
    """
    urls = []
    # eBOSS / BOSS reductions (DR17)
    for v in ["v5_13_2", "v5_13_0", "v5_10_10", "v5_7_0"]:
        urls.append(
            f"https://data.sdss.org/sas/dr17/eboss/spectro/redux/{v}/"
            f"spectra/lite/{plate}/spec-{plate}-{mjd:05d}-{fiber:04d}.fits"
        )
    # SDSS-I/II legacy (run2d=26, plate em 4-digit zero-padded)
    urls.append(
        f"https://data.sdss.org/sas/dr17/sdss/spectro/redux/26/"
        f"spectra/lite/{plate:04d}/spec-{plate:04d}-{mjd:05d}-{fiber:04d}.fits"
    )
    for run2d in ["103", "104"]:
        urls.append(
            f"https://data.sdss.org/sas/dr17/sdss/spectro/redux/{run2d}/"
            f"spectra/lite/{plate:04d}/spec-{plate:04d}-{mjd:05d}-{fiber:04d}.fits"
        )
    return urls


def read_spec(plate, mjd, fiber, cache_dir, max_attempts=2, delay=1):
    """Le um espectro (cache > download). Retorna (wave, flux, sigma) ou Nones se falhar."""
    cache_path = get_cache_path(cache_dir, plate, mjd, fiber)

    # Tenta cache
    if cache_path.exists():
        try:
            with fits.open(cache_path) as f:
                d = f[1].data
                W = 10 ** d['loglam']
                F = d['flux'] * 1e-17
                iVar = d['ivar']
                ok = np.argwhere(iVar > 0).flatten()
                if len(ok) == 0:
                    return None, None, None
                return W[ok], F[ok], 1e-17 / np.sqrt(iVar[ok])
        except Exception:
            cache_path.unlink(missing_ok=True)

    # Tenta cada URL candidata
    for url in candidate_urls(plate, mjd, fiber):
        for attempt in range(max_attempts):
            try:
                with fits.open(url) as f:
                    d = f[1].data
                    f.writeto(cache_path, overwrite=True)
                W = 10 ** d['loglam']
                F = d['flux'] * 1e-17
                iVar = d['ivar']
                ok = np.argwhere(iVar > 0).flatten()
                if len(ok) == 0:
                    return None, None, None
                return W[ok], F[ok], 1e-17 / np.sqrt(iVar[ok])
            except Exception as e:
                if "404" in str(e):
                    break  # passa pra proxima URL
                time.sleep(delay)

    return None, None, None


## 6. Detectar nomes de colunas

Os FITS do SDSS as vezes usam variantes de nome (`PLATE` vs `plate`,
`Z` vs `z_pipeline`). Detectamos automaticamente.


In [19]:
# Nomes exatos das colunas no FITS full do eBOSS QSO DR16
COL_PLATE             = "PLATE"
COL_MJD               = "MJD"
COL_FIBERID           = "FIBERID"
COL_RA                = "RA"
COL_DEC               = "DEC"
COL_IS_QSO            = "IS_QSO_FINAL"     # boolean — confirma QSO

# Redshift principal e qualidade
COL_REDSHIFT          = "Z"
COL_ZWARNING          = "ZWARNING"

# Qualidade do espectro
COL_SN_MEDIAN_ALL     = "SN_MEDIAN_ALL"

# Fotometria optica (MODELMAG vetor [u,g,r,i,z])
COL_MODELMAG          = "MODELMAG"

# Fotometria IR (WISE)
COL_W1_MAG            = "W1_MAG"
COL_W1_NMAG           = "W1_NANOMAGGIES"
COL_W2_NMAG           = "W2_NANOMAGGIES"

# === Estimativas alternativas de redshift ===
COL_Z_PCA             = "Z_PCA"
COL_DELTACHI2_PCA     = "DELTACHI2_PCA"
COL_ZWARN_PCA         = "ZWARN_PCA"

COL_Z_VI              = "Z_VI"
COL_Z_DR12Q           = "Z_DR12Q"

# === Redshifts por linha de emissao ===
LINE_COLUMNS = {
    "lya":    ("Z_LYA",    "DELTACHI2_LYA",    "ZWARN_LYA"),
    "civ":    ("Z_CIV",    "DELTACHI2_CIV",    "ZWARN_CIV"),
    "ciii":   ("Z_CIII",   "DELTACHI2_CIII",   "ZWARN_CIII"),
    "mgii":   ("Z_MGII",   "DELTACHI2_MGII",   "ZWARN_MGII"),
    "hbeta":  ("Z_HBETA",  "DELTACHI2_HBETA",  "ZWARN_HBETA"),
    "halpha": ("Z_HALPHA", "DELTACHI2_HALPHA", "ZWARN_HALPHA"),
}

# Sanity check
required = [COL_PLATE, COL_MJD, COL_FIBERID, COL_RA, COL_DEC, COL_REDSHIFT]
missing = [c for c in required if c not in filtered.colnames]
if missing:
    raise ValueError(f"Colunas obrigatorias nao encontradas: {missing}")

optional = [COL_IS_QSO, COL_ZWARNING, COL_SN_MEDIAN_ALL,
            COL_MODELMAG, COL_W1_MAG, COL_W1_NMAG, COL_W2_NMAG,
            COL_Z_PCA, COL_DELTACHI2_PCA, COL_ZWARN_PCA,
            COL_Z_VI, COL_Z_DR12Q]
for cols in LINE_COLUMNS.values():
    optional.extend(cols)

missing_opt = [c for c in optional if c and c not in filtered.colnames]
if missing_opt:
    print(f"[aviso] Colunas opcionais ausentes (serao NaN): {missing_opt}")
else:
    print("Todas as colunas encontradas.")

if COL_MODELMAG in filtered.colnames:
    print(f"MODELMAG shape: {filtered[COL_MODELMAG].shape}  (esperado: (N, 5))")


Todas as colunas encontradas.
MODELMAG shape: (343708, 5)  (esperado: (N, 5))


## 7. Filtrar entradas validas

Aplica cortes de qualidade:
- `Z >= 0` (redshift fisico)
- `ZWARNING == 0` se quiser apenas confiaveis (controlado por flag)


In [ ]:
# IS_QSO_FINAL==1 descartava 268k QSO BONS (Z identico ao do catalogo de clustering,
# std 0.000) -> e' corte de pipeline desnecessario; a amostra de clustering ja e' a
# confirmacao. Mantemos so ZWARNING==0 (label limpo) -> ~306k QSO. Ver Conclusao 4.
USE_QSO_CUT       = False              # NAO filtra IS_QSO_FINAL (excluia QSO bons)
USE_ZWARNING_CUT  = True               # mantem so zwarning == 0 (label limpo; remove sentinelas 999999 e flagados)
N_LIMIT           = None               # None = todos (~306k). Para teste rapido use 1000.

mask = filtered[COL_REDSHIFT] >= 0
print(f"Apos z >= 0:                {mask.sum():,}/{len(filtered):,}")

if USE_QSO_CUT and COL_IS_QSO in filtered.colnames:
    n_before = mask.sum()
    mask &= (filtered[COL_IS_QSO] == 1)
    n_removed = n_before - mask.sum()
    print(f"Apos IS_QSO_FINAL == 1:     {mask.sum():,} (removidos {n_removed} nao-QSO)")

if USE_ZWARNING_CUT:
    mask &= filtered[COL_ZWARNING] == 0
    print(f"Apos ZWARNING == 0:         {mask.sum():,}")

cat = filtered[mask]
if N_LIMIT is not None:
    cat = cat[:N_LIMIT]
    print(f"Limitado para {N_LIMIT:,} (TESTE)")

print(f"\nFinal: {len(cat):,} objetos a processar")


## 8. Baixar/ler espectros em paralelo

Usa `ProcessPoolExecutor` para paralelizar. Ajuste `MAX_WORKERS` conforme
a maquina (default = numero de CPUs).


In [23]:
MAX_WORKERS = max(4, os.cpu_count() - 2)
print(f"Workers: {MAX_WORKERS}")

# Prepara lista de tarefas
tasks = [(int(cat[COL_PLATE][i]),
          int(cat[COL_MJD][i]),
          int(cat[COL_FIBERID][i]),
          paths["cache_dir"]) for i in range(len(cat))]


def worker(args):
    """Le espectro e computa S/N mediano."""
    plate, mjd, fiber, cache_dir = args
    W, F, S = read_spec(plate, mjd, fiber, cache_dir)
    if W is None:
        return (plate, mjd, fiber, None, None, None, None)
    # S/N mediano: F / sigma = flux * sqrt(ivar)
    sn = float(np.median(F / S)) if len(F) else float("nan")
    return (plate, mjd, fiber, W, F, S, sn)


# Testa com 1 espectro primeiro
print("\nTeste com 1 espectro:")
test_plate, test_mjd, test_fiber, _ = tasks[0]
W, F, S = read_spec(test_plate, test_mjd, test_fiber, paths["cache_dir"])
if W is None:
    print(f"  [FALHA] {test_plate}-{test_mjd}-{test_fiber}")
    raise RuntimeError("Teste falhou — verifique conectividade ou primeiro espectro")
print(f"  [OK] {test_plate}-{test_mjd}-{test_fiber}: {len(W)} pts, "
      f"{W[0]:.0f}-{W[-1]:.0f} A, S/N med = {np.median(F / S):.2f}")


Workers: 18

Teste com 1 espectro:
  [OK] 441-51868-1: 3800 pts, 3813-9145 A, S/N med = 5.86


In [24]:
# Loop principal — processa todos os espectros
results = {}  # spec_idx -> (wave, flux, sigma, sn_median)
fails = 0

with ProcessPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futures = {ex.submit(worker, tasks[i]): i for i in range(len(tasks))}
    for fut in tqdm(as_completed(futures), total=len(tasks), desc="Espectros"):
        i = futures[fut]
        try:
            plate, mjd, fiber, W, F, S, sn = fut.result()
            if W is None:
                fails += 1
                continue
            results[i] = (W.astype(np.float32),
                          F.astype(np.float32),
                          S.astype(np.float32),
                          sn)
        except Exception as e:
            fails += 1
            print(f"[erro] indice {i}: {e}")

print(f"\nSucesso: {len(results):,}/{len(tasks):,}")
print(f"Falhas:  {fails:,}")


Espectros: 100%|██████████| 100/100 [00:19<00:00,  5.19it/s]


Sucesso: 100/100
Falhas:  0


## 9. Salvar HDF5 estruturado

In [25]:
with h5py.File(paths["spectra_h5"], "w") as f:
    valid_idx = sorted(results.keys())
    N = len(valid_idx)
    print(f"Salvando {N:,} espectros em {paths['spectra_h5']}")

    spec_ids = [f"{OBJECT_TYPE}_{i:06d}" for i in range(N)]

    def col_or_nan(col_name, dtype=np.float32):
        if not col_name or col_name not in filtered.colnames:
            return np.full(N, np.nan, dtype=dtype)
        return np.array([cat[col_name][i] for i in valid_idx], dtype=dtype)

    def modelmag_band(band_idx, dtype=np.float32):
        """0=u, 1=g, 2=r, 3=i, 4=z."""
        if COL_MODELMAG not in filtered.colnames:
            return np.full(N, np.nan, dtype=dtype)
        return np.array([cat[COL_MODELMAG][i, band_idx] for i in valid_idx], dtype=dtype)

    cat_grp = f.create_group("catalog")

    # ID
    cat_grp.create_dataset("spec_id",          data=np.array(spec_ids, dtype="S20"))

    # Identificacao
    cat_grp.create_dataset("plate",            data=col_or_nan(COL_PLATE, np.int32))
    cat_grp.create_dataset("mjd",              data=col_or_nan(COL_MJD, np.int32))
    cat_grp.create_dataset("fiberid",          data=col_or_nan(COL_FIBERID, np.int32))

    # Posicao
    cat_grp.create_dataset("ra",               data=col_or_nan(COL_RA, np.float64))
    cat_grp.create_dataset("dec",              data=col_or_nan(COL_DEC, np.float64))

    # Classificacao
    cat_grp.create_dataset("is_qso_final",     data=col_or_nan(COL_IS_QSO, np.int32))

    # Redshift principal
    cat_grp.create_dataset("redshift",         data=col_or_nan(COL_REDSHIFT, np.float32))
    cat_grp.create_dataset("zwarning",         data=col_or_nan(COL_ZWARNING, np.int32))

    # Qualidade do espectro
    cat_grp.create_dataset("sn_median",        data=np.array([results[i][3] for i in valid_idx],
                                                              dtype=np.float32))
    cat_grp.create_dataset("sn_median_all",    data=col_or_nan(COL_SN_MEDIAN_ALL, np.float32))

    # Fotometria optica
    g_arr = modelmag_band(1)
    r_arr = modelmag_band(2)
    i_arr = modelmag_band(3)
    z_arr = modelmag_band(4)
    cat_grp.create_dataset("g_mag",            data=g_arr)
    cat_grp.create_dataset("r_mag",            data=r_arr)
    cat_grp.create_dataset("i_mag",            data=i_arr)
    cat_grp.create_dataset("z_mag",            data=z_arr)
    cat_grp.create_dataset("gr_color",         data=(g_arr - r_arr).astype(np.float32))
    cat_grp.create_dataset("ri_color",         data=(r_arr - i_arr).astype(np.float32))

    # Fotometria IR
    cat_grp.create_dataset("w1_mag",           data=col_or_nan(COL_W1_MAG, np.float32))
    w1n = col_or_nan(COL_W1_NMAG, np.float64)
    w2n = col_or_nan(COL_W2_NMAG, np.float64)
    with np.errstate(invalid="ignore", divide="ignore"):
        w1w2 = (-2.5 * np.log10(w1n / w2n)).astype(np.float32)
    cat_grp.create_dataset("w1_w2_color",      data=w1w2)

    # BONUS: estimativas alternativas de redshift
    cat_grp.create_dataset("z_pca",            data=col_or_nan(COL_Z_PCA, np.float32))
    cat_grp.create_dataset("deltachi2_pca",    data=col_or_nan(COL_DELTACHI2_PCA, np.float32))
    cat_grp.create_dataset("zwarn_pca",        data=col_or_nan(COL_ZWARN_PCA, np.int32))
    cat_grp.create_dataset("z_vi",             data=col_or_nan(COL_Z_VI, np.float32))
    cat_grp.create_dataset("z_dr12q",          data=col_or_nan(COL_Z_DR12Q, np.float32))

    # BONUS: redshifts por linha de emissao
    for line, (col_z, col_dchi2, col_zwarn) in LINE_COLUMNS.items():
        cat_grp.create_dataset(f"z_{line}",          data=col_or_nan(col_z, np.float32))
        cat_grp.create_dataset(f"deltachi2_{line}",  data=col_or_nan(col_dchi2, np.float32))
        cat_grp.create_dataset(f"zwarn_{line}",      data=col_or_nan(col_zwarn, np.int32))

    # ---- spectra/ : 1 grupo por espectro com wave, flux, ivar ----
    spec_grp = f.create_group("spectra")
    for new_i, orig_i in enumerate(tqdm(valid_idx, desc="Escrevendo HDF5")):
        sid = spec_ids[new_i]
        W, F, S, _ = results[orig_i]
        g = spec_grp.create_group(sid)
        g.create_dataset("wave", data=W, compression="gzip", compression_opts=4)
        g.create_dataset("flux", data=F, compression="gzip", compression_opts=4)
        g.create_dataset("ivar", data=(1.0 / (S ** 2)).astype(np.float32),
                         compression="gzip", compression_opts=4)

print(f"\nHDF5 escrito: {paths['spectra_h5']} "
      f"({paths['spectra_h5'].stat().st_size / 1e9:.2f} GB)")


Salvando 100 espectros em /home/thalita/spec_z_ml/data/processed/QSO/QSOspectra_all.h5


Escrevendo HDF5: 100%|██████████| 100/100 [00:00<00:00, 902.63it/s]


HDF5 escrito: /home/thalita/spec_z_ml/data/processed/QSO/QSOspectra_all.h5 (0.01 GB)


## 10. Verificacao

Reabre o HDF5 e checa estrutura.


In [26]:
with h5py.File(paths["spectra_h5"], "r") as f:
    cat = f["catalog"]
    print(f"Grupos: {list(f.keys())}")
    print(f"\ncatalog/ datasets ({len(cat)}): {list(cat.keys())}")
    print(f"  N: {len(cat['spec_id']):,}")
    print(f"  z range:        {cat['redshift'][:].min():.3f} - {cat['redshift'][:].max():.3f}")
    print(f"  S/N mediano:    {np.nanmedian(cat['sn_median'][:]):.2f} (computado)")
    print(f"  S/N pipeline:   {np.nanmedian(cat['sn_median_all'][:]):.2f}")

    print(f"\n  Fotometria optica:")
    print(f"    g_mag:        {np.nanmin(cat['g_mag'][:]):.2f} - {np.nanmax(cat['g_mag'][:]):.2f}")
    print(f"    gr_color:     {np.nanmin(cat['gr_color'][:]):.2f} - {np.nanmax(cat['gr_color'][:]):.2f}")
    print(f"    ri_color:     {np.nanmin(cat['ri_color'][:]):.2f} - {np.nanmax(cat['ri_color'][:]):.2f}")
    print(f"    w1_mag:       {np.nanmin(cat['w1_mag'][:]):.2f} - {np.nanmax(cat['w1_mag'][:]):.2f}")
    print(f"    w1_w2_color:  {np.nanmin(cat['w1_w2_color'][:]):.2f} - {np.nanmax(cat['w1_w2_color'][:]):.2f}")

    n_zwarn = int((cat['zwarning'][:] != 0).sum())
    print(f"\n  ZWARNING != 0:  {n_zwarn} ({100*n_zwarn/len(cat['zwarning']):.1f}%)")

    is_qso = int((cat['is_qso_final'][:] == 1).sum())
    print(f"  IS_QSO_FINAL=1: {is_qso} ({100*is_qso/len(cat['is_qso_final']):.1f}%)")

    # Comparacao entre estimativas de redshift
    print(f"\n  === Multiplas estimativas de redshift ===")
    z_main = cat['redshift'][:]
    for alt_name, alt_key in [("Z_PCA",   "z_pca"),
                               ("Z_VI",    "z_vi"),
                               ("Z_DR12Q", "z_dr12q")]:
        z_alt = cat[alt_key][:]
        valid = ~np.isnan(z_main) & ~np.isnan(z_alt) & (z_alt > 0)
        if valid.sum() > 0:
            diff = z_main[valid] - z_alt[valid]
            print(f"    Z - {alt_name}: mediana={np.median(diff):+.4f}, "
                  f"std={np.std(diff):.4f} (n={valid.sum()})")

    print(f"\n  === Redshifts por linha de emissao ===")
    for line in ["lya", "civ", "ciii", "mgii", "hbeta", "halpha"]:
        z_line = cat[f"z_{line}"][:]
        valid_line = ~np.isnan(z_line) & (z_line > 0)
        if valid_line.sum() > 0:
            diff = z_main[valid_line] - z_line[valid_line]
            print(f"    Z - Z_{line.upper():6s}: mediana={np.median(diff):+.4f} "
                  f"(n={valid_line.sum()})")

    print(f"\nspectra/: {len(f['spectra']):,} grupos")
    sample_id = list(f["spectra"].keys())[0]
    g = f[f"spectra/{sample_id}"]
    print(f"  Exemplo {sample_id}: wave {g['wave'].shape}  "
          f"({g['wave'][0]:.0f}-{g['wave'][-1]:.0f} A)")


Grupos: ['catalog', 'spectra']

catalog/ datasets (42): ['dec', 'deltachi2_ciii', 'deltachi2_civ', 'deltachi2_halpha', 'deltachi2_hbeta', 'deltachi2_lya', 'deltachi2_mgii', 'deltachi2_pca', 'fiberid', 'g_mag', 'gr_color', 'i_mag', 'is_qso_final', 'mjd', 'plate', 'r_mag', 'ra', 'redshift', 'ri_color', 'sn_median', 'sn_median_all', 'spec_id', 'w1_mag', 'w1_w2_color', 'z_ciii', 'z_civ', 'z_dr12q', 'z_halpha', 'z_hbeta', 'z_lya', 'z_mag', 'z_mgii', 'z_pca', 'z_vi', 'zwarn_ciii', 'zwarn_civ', 'zwarn_halpha', 'zwarn_hbeta', 'zwarn_lya', 'zwarn_mgii', 'zwarn_pca', 'zwarning']
  N: 100
  z range:        0.801 - 2.200
  S/N mediano:    6.77 (computado)
  S/N pipeline:   4.82

  Fotometria optica:
    g_mag:        18.06 - 21.53
    gr_color:     -0.27 - 1.17
    ri_color:     -0.17 - 0.62
    w1_mag:       14.16 - 17.59
    w1_w2_color:  0.58 - 1.85

  ZWARNING != 0:  61 (61.0%)
  IS_QSO_FINAL=1: 100 (100.0%)

  === Multiplas estimativas de redshift ===
    Z - Z_PCA: mediana=+0.0000, std=0.000